In [1]:
import boto3
import json
import uuid
import pandas as pd
from datetime import datetime

# Configuration
INPUT_BUCKET = "test-passport-data"
BLUEPRINT_ARN = "arn:aws:bedrock:us-east-1:122610487956:blueprint/14e3e065923e"
REGION_NAME = "us-east-1"
OUTPUT_S3_PATH = "s3://test-passport-data/results/"

# Initialize clients
client = boto3.client('bedrock-data-automation-runtime', region_name=REGION_NAME)
s3_client = boto3.client('s3', region_name=REGION_NAME)

# Expected passport fields from your blueprint
PASSPORT_FIELDS = {
    'country': 'The country code of the passport issuing country',
    'documentType': 'The type of document (PASSPORT)', 
    'surname': 'The last name or family name',
    'documentNumber': 'The passport document number',
    'issuingOffice': 'The issuing office or authority',
    'validFrom': 'The date from which passport is valid',
    'forenames': 'The first name(s) or given name(s)',
    'validTo': 'The expiration date of the passport'
}

print(" Setup complete with passport field mapping")


 Setup complete with passport field mapping


In [2]:
def extract_passport_data(file_key):
    """Extract passport data and return structured results"""
    token = f"test{uuid.uuid4().hex[:10]}"
    
    # Invoke BDA
    params = {
        'clientToken': token,
        'inputConfiguration': {
            's3Uri': f"s3://{INPUT_BUCKET}/{file_key}"
        },
        'outputConfiguration': {
            's3Uri': f"{OUTPUT_S3_PATH}{token}/"
        },
        'blueprints': [{
            'blueprintArn': BLUEPRINT_ARN,
            'stage': 'DEVELOPMENT'
        }],
        'dataAutomationProfileArn': f'arn:aws:bedrock:us-east-1:122610487956:data-automation-profile/us.data-automation-v1'
    }
    
    try:
        # Start processing
        response = client.invoke_data_automation_async(**params)
        invocation_arn = response['invocationArn']
        
        print(f"✅ Processing started: {invocation_arn}")
        
        # Wait for completion
        import time
        while True:
            status_response = client.get_data_automation_status(invocationArn=invocation_arn)
            status = status_response['status']
            
            print(f"Status: {status}")
            
            if status == 'Success':
                # Get results from S3
                output_uri = status_response['outputConfiguration']['s3Uri']
                results = get_extraction_results(output_uri)
                return parse_passport_fields(results, file_key)
                
            elif status == 'Failed':
                return {'error': 'Processing failed', 'file_key': file_key}
                
            time.sleep(5)  # Wait 5 seconds before checking again
            
    except Exception as e:
        print(f"❌ Error: {e}")
        return {'error': str(e), 'file_key': file_key}

def get_extraction_results(s3_uri):
    """Retrieve extraction results from S3"""
    # Parse S3 URI
    uri_parts = s3_uri.replace('s3://', '').split('/', 1)
    bucket = uri_parts[0]
    key = uri_parts[1]
    
    # Get the metadata file
    response = s3_client.get_object(Bucket=bucket, Key=key)
    return json.loads(response['Body'].read().decode('utf-8'))

def parse_passport_fields(extraction_results, file_key):
    """Parse passport fields from BDA results"""
    passport_data = {
        'file_key': file_key,
        'processing_timestamp': datetime.now().isoformat(),
        'extraction_status': 'success'
    }
    
    # Initialize all expected fields
    for field in PASSPORT_FIELDS.keys():
        passport_data[field] = None
    
    try:
        # Navigate through BDA output structure
        for segment in extraction_results.get('output_metadata', []):
            for seg_metadata in segment.get('segment_metadata', []):
                if seg_metadata.get('custom_output_status') == 'MATCH':
                    custom_output_path = seg_metadata['custom_output_path']
                    custom_results = get_extraction_results(custom_output_path)
                    
                    # Extract inference results
                    inference_result = custom_results.get('inference_result', {})
                    
                    # Map each field from your blueprint
                    for field in PASSPORT_FIELDS.keys():
                        if field in inference_result:
                            passport_data[field] = inference_result[field]
                    
                    break
                    
    except Exception as e:
        passport_data['extraction_status'] = 'error'
        passport_data['error_message'] = str(e)
    
    return passport_data

print(" Enhanced extraction functions ready")


 Enhanced extraction functions ready


In [10]:
# Test with passport file
test_file = "test_passport/china.jpg"  # Your Chinese passport file
result = extract_passport_data(test_file)

print("EXTRACTED PASSPORT DATA:")
print("=" * 50)
for field, value in result.items():
    if field in PASSPORT_FIELDS:
        print(f"{field:15}: {value}")

print("\n📋 Full Result:")
print(json.dumps(result, indent=2, default=str))


✅ Processing started: arn:aws:bedrock:us-east-1:122610487956:data-automation-invocation/a31c9bfb-c03e-4338-932d-6168c8f58303
Status: InProgress
Status: InProgress
Status: InProgress
Status: Success
EXTRACTED PASSPORT DATA:
country        : PEOPLE'S REPUBLIC OF CHINA
documentType   : SERVICE PASSPORT
surname        : YANG
documentNumber : SE0000000
issuingOffice  : MINISTRY OF FOREIGN AFFAIRS
validFrom      : 2010-05-01
forenames      : ZHAO
validTo        : 2015-05-01

📋 Full Result:
{
  "file_key": "test_passport/china.jpg",
  "processing_timestamp": "2025-08-27T15:32:09.441562",
  "extraction_status": "success",
  "country": "PEOPLE'S REPUBLIC OF CHINA",
  "documentType": "SERVICE PASSPORT",
  "surname": "YANG",
  "documentNumber": "SE0000000",
  "issuingOffice": "MINISTRY OF FOREIGN AFFAIRS",
  "validFrom": "2010-05-01",
  "forenames": "ZHAO",
  "validTo": "2015-05-01"
}


In [3]:
def process_all_passports():
    """Process all passport files and store results"""
    
    # Get list of files
    response = s3_client.list_objects_v2(Bucket=INPUT_BUCKET, Prefix="test_passport/")
    files = [obj['Key'] for obj in response.get('Contents', []) if not obj['Key'].endswith('/')]
    
    results = []
    print(f"🚀 Processing {len(files)} passport files...")
    
    for i, file_key in enumerate(files, 1):
        print(f"\n📄 Processing {i}/{len(files)}: {file_key}")
        result = extract_passport_data(file_key)
        results.append(result)
        
        # Show extracted data
        if result.get('extraction_status') == 'success':
            print(f"✅ Extracted: {result.get('forenames', 'N/A')} {result.get('surname', 'N/A')}")
            print(f"   Country: {result.get('country', 'N/A')}, Doc#: {result.get('documentNumber', 'N/A')}")
        else:
            print(f"❌ Failed: {result.get('error_message', 'Unknown error')}")
    
    return results

# Run batch processing
all_results = process_all_passports()


🚀 Processing 31 passport files...

📄 Processing 1/31: test_passport/100035732_20250827_165707_58b9e18b.jpg
✅ Processing started: arn:aws:bedrock:us-east-1:122610487956:data-automation-invocation/ac9440b7-d57c-4965-81e2-00df97067264
Status: InProgress
Status: InProgress
Status: InProgress
Status: Success
✅ Extracted: SAKURA GAIMU
   Country: JAPAN, Doc#: XS1234567

📄 Processing 2/31: test_passport/153547_20250827_165709_7f26cc29.jpg
✅ Processing started: arn:aws:bedrock:us-east-1:122610487956:data-automation-invocation/20a39f8b-0d49-4fe5-a4e7-8523b5163cd5
Status: InProgress
Status: InProgress
Status: InProgress
Status: Success
✅ Extracted: ZEYNEP ORNEK
   Country: TUR, Doc#: P 00100366

📄 Processing 3/31: test_passport/2517D61900000578-2929980-A_prototype_of_the_new_Irish_Passport_Card_issued_by_Ireland_s_D-a-22_1422461631989_20250827_165709_1a83179d.jpg
✅ Processing started: arn:aws:bedrock:us-east-1:122610487956:data-automation-invocation/8fe74666-6a29-42ee-a06f-c5b69fb6a018
Status: I